# <b><font color='cornflowerblue'>Сюжет игры</font></b>
Ночью из дата-центра исчез файл.

Логи повреждены. Известно только, что:
- саботаж совершил сотрудник компании;
- он оставил следы в данных;

Восстановить доступ можно, если определить:
- город
- отдел
- сотрудника
- устройство
- дату атаки
- код операции

Из этих шести частей собирается пароль.
Пароль является кодом доступа к исчезнувшему файлу.

<b> Задача: </b> Восстановить пароль и получить доступ к файлу.

## <b><font color='cornflowerblue'>Данные</font></b>
- employees.csv
- devices.csv
- incidents.csv
- access_logs.csv

In [2]:
import pandas as pd
import numpy as np


In [3]:
# - employees.csv
# - devices.csv
# - incidents.csv
# - access_logs.csv

employees = pd.read_csv('..//data//employees.csv')
display(employees)
devices = pd.read_csv('..//data//devices.csv')
display(devices)
incidents = pd.read_csv('..//data//incidents.csv')
display(incidents)
access_logs = pd.read_csv('..//data//access_logs.csv')
display(access_logs)


,Unnamed: 0,employee_id,name,city,department,experience,salary
0,0,1151,James Miller,Berlin,Security,12,125000
1,1,1002,Benjamin Davis,Rome,Security,8,94392
2,2,1003,Henry Thomas,Madrid,IT,1,92966
3,3,1004,Emma White,Rome,Finance,13,99325
4,4,1005,Sophia Harris,Paris,IT,10,84230
...,...,...,...,...,...,...,...
145,145,1146,James Young,Madrid,IT,6,76279
146,146,1147,Oliver Miller,Rome,Operations,13,86227
147,147,1148,Mia Walker,Madrid,Finance,10,128901
148,148,1149,Ethan Thomas,Madrid,IT,9,93459


,Unnamed: 0,device_id,device_type,security_level,location
0,0,1,SERVER-01,medium,London
1,1,2,SERVER-02,low,Madrid
2,2,3,SERVER-03,critical,Rome
3,3,4,SERVER-04,critical,Rome
4,4,5,SERVER-05,medium,Amsterdam
5,5,6,SERVER-06,low,Amsterdam
6,6,7,SERVER-07,critical,Berlin
7,7,8,SERVER-08,high,Madrid
8,8,9,SERVER-09,medium,Rome
9,9,10,SERVER-10,high,Berlin


,Unnamed: 0,employee_id,incident_type,severity,date
0,0,1124,privilege_escalation,high,2026-05-07
1,1,1083,policy_violation,low,2026-05-06
2,2,1134,policy_violation,medium,2026-05-21
3,3,1081,data_copy,low,2026-05-19
4,4,1002,privilege_escalation,high,2026-05-14
...,...,...,...,...,...
520,520,1051,policy_violation,high,2026-05-13
521,521,1051,data_copy,high,2026-05-11
522,522,1051,privilege_escalation,high,2026-05-12
523,523,1051,policy_violation,high,2026-05-11


,Unnamed: 0,employee_id,device_id,date,action,duration
0,0,1044,20,2026-05-08,DELETE,14
1,1,1147,26,2026-05-01,LOGIN,278
2,2,1049,27,2026-05-31,DELETE,59
3,3,1091,14,2026-05-01,DELETE,415
4,4,1081,36,2026-05-23,EXPORT,316
...,...,...,...,...,...,...
2555,2555,1051,7,2026-05-14,LOGIN,60
2556,2556,1051,7,2026-05-14,LOGIN,177
2557,2557,1051,7,2026-05-14,LOGIN,178
2558,2558,1051,7,2026-05-14,LOGIN,58


## <b><font color='cornflowerblue'>Глава 1. Подозрительные города</font></b>
<u>Система сообщает:</u> Атака произошла из города с максимальным числом инцидентов высокой критичности.

<i>1 часть пароля:</i> 3 первых буквы получившегося города

In [4]:
employees[["experience", "city"]].max()
# Rom

experience      14
city          Rome
dtype: object

In [5]:
employees

,Unnamed: 0,employee_id,name,city,department,experience,salary
0,0,1151,James Miller,Berlin,Security,12,125000
1,1,1002,Benjamin Davis,Rome,Security,8,94392
2,2,1003,Henry Thomas,Madrid,IT,1,92966
3,3,1004,Emma White,Rome,Finance,13,99325
4,4,1005,Sophia Harris,Paris,IT,10,84230
...,...,...,...,...,...,...,...
145,145,1146,James Young,Madrid,IT,6,76279
146,146,1147,Oliver Miller,Rome,Operations,13,86227
147,147,1148,Mia Walker,Madrid,Finance,10,128901
148,148,1149,Ethan Thomas,Madrid,IT,9,93459


In [6]:
incidents_2 = incidents.merge(employees, on = 'employee_id', how = 'left')
display(incidents_2)

,Unnamed: 0_x,employee_id,incident_type,severity,date,Unnamed: 0_y,name,city,department,experience,salary
0,0,1124,privilege_escalation,high,2026-05-07,123,Amelia Miller,Rome,Security,11,119063
1,1,1083,policy_violation,low,2026-05-06,82,Oliver Brown,Berlin,Security,1,108700
2,2,1134,policy_violation,medium,2026-05-21,133,Emma Hall,Paris,IT,3,82609
3,3,1081,data_copy,low,2026-05-19,80,Ethan King,Amsterdam,Security,9,67530
4,4,1002,privilege_escalation,high,2026-05-14,1,Benjamin Davis,Rome,Security,8,94392
...,...,...,...,...,...,...,...,...,...,...,...
520,520,1051,policy_violation,high,2026-05-13,50,James Walker,London,HR,11,101164
521,521,1051,data_copy,high,2026-05-11,50,James Walker,London,HR,11,101164
522,522,1051,privilege_escalation,high,2026-05-12,50,James Walker,London,HR,11,101164
523,523,1051,policy_violation,high,2026-05-11,50,James Walker,London,HR,11,101164


In [7]:
incidents_2_g = incidents_2.groupby(['severity', 'city']).agg({'employee_id': 'count'}).sort_values('employee_id', ascending = False)
display(incidents_2_g)
# Lon

employee_id
severity city                  
high     London              40
         Berlin              38
medium   Amsterdam           37
high     Madrid              36
low      Madrid              35
         Berlin              33
         Rome                32
high     Rome                32
         Amsterdam           31
medium   Berlin              31
low      London              30
         Amsterdam           29
         Paris               29
high     Paris               20
medium   Madrid              20
         Paris               20
         London              16
         Rome                16

In [8]:
# Часть 1: 3 первые буквы города с максимальным числом инцидентов высокой критичности
top_city = incidents_2_g.reset_index().iloc[0]['city']
part1 = top_city[:3].upper()
print(f'Город: {top_city}')
print(f'Часть 1 пароля: {part1}')


Город: London
Часть 1 пароля: LON


## <b><font color='cornflowerblue'>Глава 2. Внутренний агент</font></b>
<u>Система сообщает:</u> Подозреваемый работает в отделе, где средняя длительность доступа к системе была максимальной.

<i>2 часть пароля:</i> 2 первых буквы названия отдела

In [9]:
access_logs

,Unnamed: 0,employee_id,device_id,date,action,duration
0,0,1044,20,2026-05-08,DELETE,14
1,1,1147,26,2026-05-01,LOGIN,278
2,2,1049,27,2026-05-31,DELETE,59
3,3,1091,14,2026-05-01,DELETE,415
4,4,1081,36,2026-05-23,EXPORT,316
...,...,...,...,...,...,...
2555,2555,1051,7,2026-05-14,LOGIN,60
2556,2556,1051,7,2026-05-14,LOGIN,177
2557,2557,1051,7,2026-05-14,LOGIN,178
2558,2558,1051,7,2026-05-14,LOGIN,58


In [10]:
employees

,Unnamed: 0,employee_id,name,city,department,experience,salary
0,0,1151,James Miller,Berlin,Security,12,125000
1,1,1002,Benjamin Davis,Rome,Security,8,94392
2,2,1003,Henry Thomas,Madrid,IT,1,92966
3,3,1004,Emma White,Rome,Finance,13,99325
4,4,1005,Sophia Harris,Paris,IT,10,84230
...,...,...,...,...,...,...,...
145,145,1146,James Young,Madrid,IT,6,76279
146,146,1147,Oliver Miller,Rome,Operations,13,86227
147,147,1148,Mia Walker,Madrid,Finance,10,128901
148,148,1149,Ethan Thomas,Madrid,IT,9,93459


In [11]:
access_logs_2 = access_logs.merge(employees, on = 'employee_id', how = 'left')
display(access_logs_2.sort_values('duration', ascending = False).head(10))

,Unnamed: 0_x,employee_id,device_id,date,action,duration,Unnamed: 0_y,name,city,department,experience,salary
1298,1298,1017,7,2026-05-25,UPLOAD,499,16,Grace Miller,Rome,HR,14,79565
1303,1303,1081,25,2026-05-14,EXPORT,499,80,Ethan King,Amsterdam,Security,9,67530
1438,1438,1070,23,2026-05-17,EXPORT,499,69,Benjamin Anderson,Berlin,HR,2,96818
2355,2355,1119,36,2026-05-29,DOWNLOAD,499,118,James Thomas,Paris,IT,3,108265
1952,1952,1133,2,2026-05-27,EXPORT,499,132,James Wilson,Amsterdam,HR,1,64216
859,859,1064,1,2026-05-06,LOGIN,498,63,Oliver Wilson,London,Security,1,97065
1941,1941,1067,18,2026-05-23,DELETE,498,66,Sophia Hall,Amsterdam,Finance,6,61974
1608,1608,1144,14,2026-05-12,DELETE,498,143,Henry Thomas,Paris,Operations,4,63176
746,746,1055,19,2026-05-06,UPLOAD,497,54,Sophia Walker,Madrid,HR,12,81289
2249,2249,1002,11,2026-05-24,DOWNLOAD,497,1,Benjamin Davis,Rome,Security,8,94392


In [12]:
# Часть 2: 2 первые буквы отдела с максимальной средней длительностью доступа
logs_with_emp = access_logs.merge(employees, on='employee_id', how='left')
dept_avg = logs_with_emp.groupby('department')['duration'].mean().sort_values(ascending=False)
print(dept_avg)

top_dept = dept_avg.idxmax()
part2 = top_dept[:2].upper()
print(f'\nОтдел: {top_dept}')
print(f'Часть 2 пароля: {part2}')


department
IT            256.261698
Finance       251.983287
Operations    250.448357
Security      249.208850
HR            237.210111
Name: duration, dtype: float64

Отдел: IT
Часть 2 пароля: IT


## <b><font color='cornflowerblue'>Глава 3. Кто именно?</font></b>
Нужно найти сотрудника, из города в первой главе, отдела во второй главе и с максимальным количеством инцидентов.

<i>3 часть пароля:</i> 3 первых буквы имени сотрудника

In [13]:
employees

,Unnamed: 0,employee_id,name,city,department,experience,salary
0,0,1151,James Miller,Berlin,Security,12,125000
1,1,1002,Benjamin Davis,Rome,Security,8,94392
2,2,1003,Henry Thomas,Madrid,IT,1,92966
3,3,1004,Emma White,Rome,Finance,13,99325
4,4,1005,Sophia Harris,Paris,IT,10,84230
...,...,...,...,...,...,...,...
145,145,1146,James Young,Madrid,IT,6,76279
146,146,1147,Oliver Miller,Rome,Operations,13,86227
147,147,1148,Mia Walker,Madrid,Finance,10,128901
148,148,1149,Ethan Thomas,Madrid,IT,9,93459


In [14]:
incidents_2 = incidents.merge(employees, on = 'employee_id', how = 'left')
display(incidents_2)

,Unnamed: 0_x,employee_id,incident_type,severity,date,Unnamed: 0_y,name,city,department,experience,salary
0,0,1124,privilege_escalation,high,2026-05-07,123,Amelia Miller,Rome,Security,11,119063
1,1,1083,policy_violation,low,2026-05-06,82,Oliver Brown,Berlin,Security,1,108700
2,2,1134,policy_violation,medium,2026-05-21,133,Emma Hall,Paris,IT,3,82609
3,3,1081,data_copy,low,2026-05-19,80,Ethan King,Amsterdam,Security,9,67530
4,4,1002,privilege_escalation,high,2026-05-14,1,Benjamin Davis,Rome,Security,8,94392
...,...,...,...,...,...,...,...,...,...,...,...
520,520,1051,policy_violation,high,2026-05-13,50,James Walker,London,HR,11,101164
521,521,1051,data_copy,high,2026-05-11,50,James Walker,London,HR,11,101164
522,522,1051,privilege_escalation,high,2026-05-12,50,James Walker,London,HR,11,101164
523,523,1051,policy_violation,high,2026-05-11,50,James Walker,London,HR,11,101164


In [15]:
# Часть 3: сотрудник из London (гл.1) и IT (гл.2) с максимальным числом инцидентов
suspects = employees[(employees['city'] == top_city) & (employees['department'] == top_dept)]
print('Сотрудники London / IT:')
print(suspects[['employee_id', 'name', 'city', 'department']])
print()

suspect_inc = incidents[incidents['employee_id'].isin(suspects['employee_id'])]
inc_count = suspect_inc.groupby('employee_id').size().sort_values(ascending=False)
print('Инциденты по сотрудникам:')
print(inc_count)

top_emp_id = inc_count.idxmax()
top_emp_name = employees[employees['employee_id'] == top_emp_id]['name'].values[0]
part3 = top_emp_name[:3].upper()
print(f'\nСотрудник: {top_emp_name} (id={top_emp_id})')
print(f'Часть 3 пароля: {part3}')


Сотрудники London / IT:
     employee_id              name    city department
62          1063        Mia Walker  London         IT
98          1099     Amelia Harris  London         IT
138         1139  Charlotte Harris  London         IT
142         1143        Noah Young  London         IT

Инциденты по сотрудникам:
employee_id
1143    6
1063    3
1139    2
dtype: int64

Сотрудник: Noah Young (id=1143)
Часть 3 пароля: NOA


## <b><font color='cornflowerblue'>Глава 4. Орудие преступления</font></b>
Атака была проведена с устройства, которое чаще всего использовалось подозреваемым.

<i>4 часть пароля:</i> номер устройства (последние 2 символа имени устройства)

In [16]:
# Часть 4: наиболее используемое устройство подозреваемого
emp_logs = access_logs[access_logs['employee_id'] == top_emp_id].copy()

device_counts = emp_logs.groupby('device_id').size().sort_values(ascending=False)
print('Использование устройств:')
print(device_counts)

top_device_id = device_counts.idxmax()
device_name = devices[devices['device_id'] == top_device_id]['device_type'].values[0]
part4 = device_name[-2:]
print(f'\nНаиболее используемое устройство: {device_name}')
print(f'Часть 4 пароля: {part4}')


Использование устройств:
device_id
3     2
8     2
6     2
2     1
4     1
9     1
10    1
11    1
12    1
13    1
14    1
15    1
17    1
18    1
19    1
20    1
23    1
24    1
26    1
31    1
34    1
dtype: int64

Наиболее используемое устройство: SERVER-03
Часть 4 пароля: 03


## <b><font color='cornflowerblue'>Глава 5. День атаки</font></b>
Файл был украден в день максимальной активности подозреваемого. 
Подсказка: ориентируйтесь на таблицу логов

<i>5 часть пароля:</i> день (например, если это 15 мая, то берем 15)

In [17]:
# Часть 5: день максимальной активности подозреваемого (по таблице логов)
emp_logs['date'] = pd.to_datetime(emp_logs['date'])
day_activity = emp_logs.groupby('date').size().sort_values(ascending=False)
print('Активность по дням:')
print(day_activity)

top_date = day_activity.idxmax()
part5 = str(top_date.day)
print(f'\nДень атаки: {top_date.strftime("%d.%m.%Y")}')
print(f'Часть 5 пароля: {part5}')


Активность по дням:
date
2026-05-01    2
2026-05-27    2
2026-05-14    2
2026-05-18    2
2026-05-05    1
2026-05-03    1
2026-05-02    1
2026-05-07    1
2026-05-13    1
2026-05-09    1
2026-05-16    1
2026-05-08    1
2026-05-17    1
2026-05-19    1
2026-05-25    1
2026-05-22    1
2026-05-26    1
2026-05-29    1
2026-05-30    1
2026-05-31    1
dtype: int64

День атаки: 01.05.2026
Часть 5 пароля: 1


## <b><font color='cornflowerblue'>Глава 6. Код операции</font></b>
Нужно:
- оставить только действия подозреваемого в день атаки;
- посчитать частоты действий;
- отсортировать;
- взять второе по популярности действие;

<i>6 часть пароля:</i> первые три символа названия операции

In [18]:
# Часть 6: второе по популярности действие подозреваемого в день атаки
day_logs = emp_logs[emp_logs['date'] == top_date]
print(f'Действия подозреваемого {top_date.strftime("%d.%m.%Y")}:')
action_counts = day_logs['action'].value_counts()
print(action_counts)
print()

second_action = action_counts.index[1]
part6 = second_action[:3].upper()
print(f'Второе по популярности действие: {second_action}')
print(f'Часть 6 пароля: {part6}')


Действия подозреваемого 01.05.2026:
action
LOGIN     1
EXPORT    1
Name: count, dtype: int64

Второе по популярности действие: EXPORT
Часть 6 пароля: EXP


## <b><font color='cornflowerblue'>Финал. Пароль</font></b>
Соберите все части пароля:
- заглавные буквы
- части пароля разделяются символом: "-"
- получившийся пароль - код доступа к архиву "Пропавший файл.zip"
- если Вы выполнили все задания верно, то архив откроется

In [19]:
# ФИНАЛ: собираем все части пароля
password = '-'.join([part1, part2, part3, part4, part5, part6])
print('=' * 50)
print(f'ПАРОЛЬ: {password}')
print('=' * 50)


ПАРОЛЬ: LON-IT-NOA-03-1-EXP
